In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('reddit_data.csv')
popular_posts = (
    df.sort_values(by="ups", ascending=False)
           .head(200)["submission_id"]
           .tolist()
)

def get_upvoted_posts_by_user(votes, test_df, subreddit="r/Showerthoughts"):
        filtered = votes[
            (votes["SUBREDDIT"] == subreddit) &
            (votes["VOTE"] == "upvote") &
            (votes["SUBMISSION_ID"].isin(set(df["submission_id"])))
        ]
        return filtered.groupby("USERNAME")["SUBMISSION_ID"].apply(set).to_dict()

actual_votes_positive = get_upvoted_posts_by_user(votes, df)

# Evaluate popularity baseline
total_ndcg, recommended_items = 0, set()
all_items = set(df['submission_id'])
user_count = 0

def ndcg_at_k(recommended, actual_upvotes, k):
    if not actual_upvotes:
        return 0.0
    # create relevance scores 
    relevance = [1 if item in actual_upvotes else 0 for item in recommended[:k]]
    # dcg
    dcg = sum((2 ** rel - 1) / np.log2(idx + 2) 
             for idx, rel in enumerate(relevance))
    #  idcg
    ideal_relevance = sorted([1] * min(len(actual_upvotes), k), reverse=True)
    idcg = sum((2 ** rel - 1) / np.log2(idx + 2) 
              for idx, rel in enumerate(ideal_relevance))

    return dcg / idcg if idcg > 0 else 0
testing_users = ['Raven2002', 'mguardian_north', 'Clen23', 'Reeses2150', 'spockspeare', 'apoeticturtle', 'Mash404', 'locks_are_paranoid', 
                 'daygloviking', 'thx1138jr', 'Livelogikal', 'MingeyMackrel', 'uncertainusurper', 
                 'lokier01', 'baddonkey', 'pierrekrahn', 'CubyChris', 'Adventurous_Guy', 'stratman42', 'VerbotenPublish']

for user in actual_votes_positive:
    if user not in testing_users:
        continue
    recs = popular_posts
    actual_positive = actual_votes_positive.get(user, set())
    ndcg = ndcg_at_k(recs, actual_positive, 200)
    print("ndcg for user", user, ndcg)
    total_ndcg += ndcg
    recommended_items.update(recs)
    user_count += 1



print("Popularity Baseline Results:")
print(f"NDCG@200: {total_ndcg / user_count if user_count else 0:.4f}")
print(f"Users evaluated: {user_count}")

NameError: name 'votes' is not defined